# 第5章 benchmark 与可信计时

**操作手册** | 热身、重复、GPU event、避免测量陷阱

本手册对应文档：`docs/part1-profiling/chapter5/index.md`  
本手册对应代码：`code/part1-profiling/chapter5/`

---

## Goal

学会设计可信 benchmark，避免伪优化陷阱。具体目标：

1. 理解 warmup、repeat、synchronize 的必要性
2. 使用 GPU event 而非 wall clock 量准 kernel 时间
3. 输出 min / median / mean / p95 / std 统计量
4. 执行三段骨架：PyTorch op、Triton kernel、HIP event 计时

## Prerequisite

- 已完成第4章 baseline benchmark
- ROCm 环境已激活（`source code/part1-profiling/activate-rocm.sh`）
- PyTorch + Triton 可用
- 理解第2-4章的 Roofline 心智模型

## Platform

本手册基于以下环境验证：

- **GPU**: AMD Radeon RX 9070 XT (gfx1201)
- **ROCm**: 7.13
- **OS**: Ubuntu 24.04 (native, kernel 6.17.0-35-generic)
- **PyTorch**: 2.11.0+rocm7.13.0
- **Triton**: 3.2.0+rocm7.13.0

其他 RDNA3/RDNA4 架构（gfx1100, gfx1151, gfx1201）均可运行，数字会有差异。

## Parameter

### 通用 benchmark 参数

| 参数 | 含义 | 推荐值 |
|------|------|--------|
| `--warmup` | 热身次数，消除首轮开销 | 20 |
| `--repeat` | 正式计时重复次数 | 200 |
| `--shape` | 输入张量形状 | 4096,4096 |
| `--dtype` | 数据类型 | fp16 / fp32 |

### 统计量说明

- **min**: 最短时间，估算带宽/算力上限时常用
- **median**: 中位数，更能代表多数情况
- **mean**: 平均值，易受异常慢的一次影响
- **p95**: 95分位数，观察尾延迟
- **std**: 标准差，判断波动是否稳定

## Execution

### 步骤1：定位仓库根目录

所有路径均相对于仓库根目录。先确认当前位置：

In [ ]:
import os
import subprocess
from pathlib import Path

# 定位仓库根目录（notebooks/part1-profiling/ -> 向上两级）
REPO_ROOT = Path.cwd().resolve().parents[1] if "notebooks" in str(Path.cwd()) else Path.cwd()
os.chdir(REPO_ROOT)
print(f"仓库根目录: {REPO_ROOT}")
print(f"当前工作目录: {Path.cwd()}")

### 步骤2：激活 ROCm 环境

In [ ]:
# 激活 ROCm 环境（如果尚未激活）
activate_script = REPO_ROOT / "code/part1-profiling/activate-rocm.sh"
if activate_script.exists():
    print(f"ROCm 激活脚本: {activate_script}")
    print("请在终端执行: source code/part1-profiling/activate-rocm.sh")
else:
    print("警告: 未找到 ROCm 激活脚本")

### 步骤3：执行骨架 A — PyTorch vector add

演示 GPU event 计时和统计量输出；本节将 bench_torch_op 的短逻辑直接放在 Notebook 中。

In [ ]:
import torch
import statistics

def bench_torch_op(shape, dtype, repeats=200, warmup=20):
    """骨架 A：PyTorch 算子计时（示例：vector add）"""
    x = torch.randn(*shape, dtype=dtype, device="cuda")
    y = torch.randn(*shape, dtype=dtype, device="cuda")
    
    # warmup：让 JIT、cache、clock 进入稳定状态
    for _ in range(warmup):
        z = x + y
    torch.cuda.synchronize()
    
    # 用 GPU event 计时，不要用 time.time()
    starts = [torch.cuda.Event(enable_timing=True) for _ in range(repeats)]
    ends = [torch.cuda.Event(enable_timing=True) for _ in range(repeats)]
    for i in range(repeats):
        starts[i].record()
        z = x + y
        ends[i].record()
    torch.cuda.synchronize()
    
    times_ms = [s.elapsed_time(e) for s, e in zip(starts, ends)]
    return {
        "mean": statistics.mean(times_ms),
        "median": statistics.median(times_ms),
        "min": min(times_ms),
        "p95": sorted(times_ms)[int(len(times_ms) * 0.95)],
        "std": statistics.pstdev(times_ms),
    }

# 执行 benchmark
if torch.cuda.is_available():
    shape = (4096, 4096)
    for dt_name, dt in [("fp32", torch.float32), ("fp16", torch.float16)]:
        stats = bench_torch_op(shape, dt)
        print(f"\n{dt_name} vector add @ {shape}:")
        print(f"  min={stats['min']:.3f} ms, median={stats['median']:.3f} ms")
        print(f"  mean={stats['mean']:.3f} ms, p95={stats['p95']:.3f} ms, std={stats['std']:.4f} ms")
else:
    print("GPU 不可用，跳过 benchmark")

### 步骤4：执行骨架 B — Triton kernel 计时与有效带宽

本节将 bench_triton_copy 的短逻辑直接放在 Notebook 中，便于逐项解释参数。

In [ ]:
try:
    import triton
    import triton.language as tl
    
    @triton.jit
    def copy_kernel(x_ptr, y_ptr, n, BLOCK: tl.constexpr):
        pid = tl.program_id(0)
        offs = pid * BLOCK + tl.arange(0, BLOCK)
        mask = offs < n
        tl.store(y_ptr + offs, tl.load(x_ptr + offs, mask=mask), mask=mask)
    
    def bench_triton_copy(n, repeats=200, warmup=20, block=1024):
        """骨架 B：Triton kernel 计时与有效带宽估算"""
        x = torch.empty(n, dtype=torch.float32, device="cuda")
        y = torch.empty_like(x)
        grid = ((n + block - 1) // block,)
        
        for _ in range(warmup):
            copy_kernel[grid](x, y, n, BLOCK=block)
        torch.cuda.synchronize()
        
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(repeats):
            copy_kernel[grid](x, y, n, BLOCK=block)
        e.record()
        torch.cuda.synchronize()
        ms = s.elapsed_time(e)
        
        # 每次迭代搬运 2 * n * 4 字节（一读一写 fp32）
        total_bytes = 2 * n * 4 * repeats
        eff_bw = total_bytes / (ms * 1e-3) / 1e9  # GB/s
        return ms, eff_bw
    
    # 执行 benchmark
    if torch.cuda.is_available():
        print("\nTriton vector copy (float32):")
        for mib in [8, 64, 256]:
            n = mib * 1024 * 1024 // 4
            ms, gbps = bench_triton_copy(n)
            print(f"  {mib:3d} MiB: time={ms:.3f} ms, eff_bw={gbps:.2f} GB/s")
    else:
        print("GPU 不可用，跳过 Triton benchmark")
        
except ImportError:
    print("Triton 不可用，跳过骨架 B")

### 步骤5：执行综合 benchmark（bench_ch4.py）

运行完整的第4章 benchmark 骨架，输出所有统计量：

In [ ]:
bench_ch4_script = REPO_ROOT / "code/part1-profiling/chapter5/bench_ch4.py"

if bench_ch4_script.exists():
    print(f"执行: python {bench_ch4_script.relative_to(REPO_ROOT)}")
    result = subprocess.run(
        ["python", str(bench_ch4_script)],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        timeout=120
    )
    print(result.stdout)
    if result.returncode != 0:
        print(f"错误: {result.stderr}")
else:
    print(f"未找到脚本: {bench_ch4_script}")

### 步骤6：统计量示例与解读

演示如何解读一组时间数据：

In [ ]:
import statistics

# 模拟一组测量数据（单位：ms）
times_ms = [0.335, 0.337, 0.336, 0.338, 0.340, 0.339, 0.337, 0.336, 0.338, 0.341]

stats = {
    "min": min(times_ms),
    "median": statistics.median(times_ms),
    "mean": statistics.mean(times_ms),
    "p95": sorted(times_ms)[int(len(times_ms) * 0.95)],
    "std": statistics.pstdev(times_ms),
}

print("统计量示例:")
print(f"  原始数据: {times_ms}")
print(f"  min={stats['min']:.3f} ms  (最短时间，估算峰值时常用)")
print(f"  median={stats['median']:.3f} ms  (中位数，代表多数情况)")
print(f"  mean={stats['mean']:.3f} ms  (平均值，易受异常影响)")
print(f"  p95={stats['p95']:.3f} ms  (95分位数，观察尾延迟)")
print(f"  std={stats['std']:.4f} ms  (标准差，判断波动)")

# 判断稳定性
cv = stats['std'] / stats['mean']  # 变异系数
print(f"\n变异系数 (CV) = {cv:.4f}")
if cv < 0.05:
    print("  -> 波动很小，测量稳定")
elif cv < 0.10:
    print("  -> 波动可接受")
else:
    print("  -> 波动较大，需检查测量方法")

## Expected Output / Interpretation

### 骨架 A 预期输出（PyTorch vector add）

在 9070XT (gfx1201) + ROCm 7.13 上，预期看到：

```
fp32 vector add @ (4096, 4096):
  min=0.335 ms, median=0.337 ms
  mean=0.338 ms, p95=0.341 ms, std=0.0020 ms

fp16 vector add @ (4096, 4096):
  min=0.171 ms, median=0.173 ms
  mean=0.174 ms, p95=0.177 ms, std=0.0018 ms
```

**解读**：
- fp16 时间约为 fp32 的一半（0.171 vs 0.335 ms）
- 但有效带宽几乎一致（~600 GB/s），说明都是 memory-bound
- std 很小（< 0.002 ms），说明测量稳定

### 骨架 B 预期输出（Triton vector copy）

```
Triton vector copy (float32):
    8 MiB: time=0.020 ms, eff_bw=785.1 GB/s
   64 MiB: time=0.223 ms, eff_bw=572.8 GB/s
  256 MiB: time=0.900 ms, eff_bw=568.6 GB/s
```

**解读**：
- 8 MiB 时带宽很高（785 GB/s），数据主要在 L2 cache
- 64 MiB 以后跌到 ~570 GB/s 并稳定，这是 GDDR6 真实带宽
- 这条曲线是后续所有 memory-bound 算子的参照线

### 关键观察点

1. **首轮特别慢**：warmup 前的第一次包含编译、缓存准备、时钟爬升
2. **median vs min**：median 更稳定，min 用于估算峰值
3. **footprint 扫描**：小输入测到 cache，大输入测到 GDDR6
4. **波动范围**：std / mean < 5% 说明测量可信

## Pass Criteria

本章操作通过标准：

### 必须满足

1. ✅ 能执行骨架 A（PyTorch vector add），输出 min / median / mean / p95 / std
2. ✅ 能执行骨架 B（Triton vector copy），输出有效带宽
3. ✅ 理解 warmup 的作用：消除首轮编译、缓存、时钟爬升开销
4. ✅ 理解 repeat 的作用：单次结果不可信，需要统计汇总
5. ✅ 理解 GPU event vs wall clock：异步提交时必须用 GPU event
6. ✅ 能解读统计量：min 估算峰值，median 代表多数，std 判断波动

### 推荐完成

7. ⭐ 运行完整的 `bench_ch4.py`，对比 fp32 / fp16 的时间和带宽
8. ⭐ 观察 footprint 扫描曲线（8 / 64 / 256 MiB），识别 L2 vs GDDR6
9. ⭐ 计算变异系数（CV = std / mean），判断测量是否稳定

### 常见问题排查

| 现象 | 可能原因 | 解决方法 |
|------|----------|----------|
| GPU 不可用 | ROCm 未激活 | `source code/part1-profiling/activate-rocm.sh` |
| Triton 导入失败 | 缺少 Python.h | `sudo apt install python3-dev` |
| 时间几乎为零 | 没有 synchronize | 检查 `torch.cuda.synchronize()` |
| 波动很大 | 后台负载 | 关闭其他 GPU 任务，增加 repeat |
| 首轮特别慢 | 正常现象 | warmup 后单独记录，不混入统计 |

---

## 延伸阅读

- [HIP Performance Guidelines](https://rocm.docs.amd.com/projects/HIP/en/latest/how-to/performance_guidelines.html)
- [PyTorch Profiler 文档](https://docs.pytorch.org/docs/stable/profiler.html)
- [ROCm Documentation](https://rocm.docs.amd.com/)

**下一章**: [第6章 用 rocprof 找到慢在哪里](./chapter6.ipynb)